# 🤖 AutoML Pilot - Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset. This template is designed to work seamlessly with datasets exported from the AutoML Pilot web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

print('Please upload your CSV dataset:')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    print(f'✅ Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head())
else:
    print('❌ Error: No file uploaded')

In [ ]:
# @title 3. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize

# Configuration
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]
recipient_email = 'your-email@example.com' # @param {type:"string"}

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    print('Comparing models...')
    best_model = cls_compare()
    leaderboard = cls_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
    print('✅ Classification model saved as best_model.pkl')
    metrics = leaderboard.iloc[0].to_dict()

else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    print('Comparing models...')
    best_model = reg_compare()
    leaderboard = reg_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')
    print('✅ Regression model saved as best_model.pkl')
    metrics = leaderboard.iloc[0].to_dict()

In [ ]:
# @title 4. Send Email Report
import smtplib
import ssl
import json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Credentials (will be injected by AutoML Pilot)
sender_email = "" 
sender_password = ""

if sender_email and sender_password and recipient_email:
    print(f'📧 Sending report to {recipient_email}...')
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"AutoML Pilot Colab Report - {task_type.capitalize()}"
        msg["From"] = sender_email
        msg["To"] = recipient_email
        
        html = f"""
        <html>
        <body>
          <h2>🚀 AutoML Pilot - Training Complete</h2>
          <p>The training on Google Colab has finished successfully.</p>
          <h3>📊 Best Model Metrics</h3>
          <pre>{json.dumps(metrics, indent=2)}</pre>
          <p>You can now download the model from Colab and upload it back to AutoML Pilot.</p>
        </body>
        </html>
        """
        msg.attach(MIMEText(html, "html"))
        
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print('✅ Email sent!')
    except Exception as e:
        print(f'❌ Failed to send email: {e}')
else:
    print('ℹ️ Email reporting skipped (credentials missing).')

In [ ]:
# @title 5. Download Model
from google.colab import files
if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
    print('✅ Triggered download for best_model.pkl')
else:
    print('❌ Model file not found.')